In [3]:
!pip  install earthengine-api geemap


In [4]:
import ee
import geemap

ee.Authenticate()
ee.Initialize()


In [5]:
import ee
import geemap

ee.Initialize()

# ----------------------------
# ÁREA DE INTERESSE
# ----------------------------
moz = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
         .filter(ee.Filter.eq('country_na', 'Mozambique'))

anos = [1991, 1996, 2001, 2006, 2011, 2016, 2021]

def get_sensor(ano):
    if ano <= 2011:
        return 'LT05'
    elif ano <= 2013:
        return 'LE07'
    else:
        return 'LC08'

def preparar_imagem(image, sensor):
    if sensor in ['LT05', 'LE07']:
        bandas = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
    else:
        bandas = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    return image.select(bandas).multiply(0.0000275).add(-0.2)

def mascara_nuvem(image):
    qa = image.select('QA_PIXEL')
    sombra = 1 << 3
    nuvem = 1 << 5
    mask = qa.bitwiseAnd(sombra).eq(0).And(qa.bitwiseAnd(nuvem).eq(0))
    return image.updateMask(mask)

def calcular_indices(img, sensor):
    if sensor in ['LT05', 'LE07']:
        b = {'blue': 'SR_B1', 'green': 'SR_B2', 'red': 'SR_B3', 'nir': 'SR_B4', 'swir1': 'SR_B5'}
    else:
        b = {'blue': 'SR_B2', 'green': 'SR_B3', 'red': 'SR_B4', 'nir': 'SR_B5', 'swir1': 'SR_B6'}

    ndvi = img.normalizedDifference([b['nir'], b['red']]).rename('NDVI')
    ndwi = img.normalizedDifference([b['green'], b['nir']]).rename('NDWI')
    mndwi = img.normalizedDifference([b['green'], b['swir1']]).rename('MNDWI')
    ndbi = img.normalizedDifference([b['swir1'], b['nir']]).rename('NDBI')
    ndbai = img.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)',
        {'SWIR1': img.select(b['swir1']), 'NIR': img.select(b['nir'])}
    ).rename('NDBaI')
    bsi = img.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))',
        {
            'SWIR1': img.select(b['swir1']),
            'RED': img.select(b['red']),
            'NIR': img.select(b['nir']),
            'BLUE': img.select(b['blue'])
        }
    ).rename('BSI')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': img.select(b['nir']),
            'RED': img.select(b['red']),
            'BLUE': img.select(b['blue'])
        }
    ).rename('EVI')
    savi = img.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': img.select(b['nir']), 'RED': img.select(b['red'])}
    ).rename('SAVI')
    ibi = img.expression(
        '(NDBI - ((NDVI + NDWI)/2))',
        {
            'NDBI': ndbi,
            'NDVI': ndvi,
            'NDWI': ndwi
        }
    ).rename('IBI')
    ui = img.expression(
        '(NDBI - NDVI) / (NDBI + NDVI)',
        {'NDBI': ndbi, 'NDVI': ndvi}
    ).rename('UI')

    return img.addBands([ndvi, ndwi, mndwi, ndbi, ndbai, bsi, evi, savi, ibi, ui])

# ----------------------------
# LOOP PRINCIPAL
# ----------------------------
for ano in anos:
    print(f'Processando {ano}...')
    sensor = get_sensor(ano)
    colecao = f'LANDSAT/{sensor}/C02/T1_L2'

    colecao_corrigida = ee.ImageCollection(colecao) \
        .filterBounds(moz) \
        .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
        .map(mascara_nuvem) \
        .map(lambda img: preparar_imagem(img, sensor))

    imagem = colecao_corrigida.median().clip(moz)

    colecao_bruta = ee.ImageCollection(colecao) \
        .filterBounds(moz) \
        .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
        .map(mascara_nuvem) \
        .median()

    imagem_indices = calcular_indices(colecao_bruta, sensor)

    # Topografia
    srtm = ee.Image("USGS/SRTMGL1_003").clip(moz).rename('elevacao')
    slope = ee.Terrain.slope(srtm).rename('declividade')
    aspect = ee.Terrain.aspect(srtm).rename('orientacao')

    # Entropia
    nir_banda = 'SR_B5' if sensor == 'LC08' else 'SR_B4'
    entropia = colecao_bruta.select(nir_banda).multiply(1000).toInt() \
        .entropy(ee.Kernel.square(2)).rename('entropia')

    imagem_final = imagem_indices \
        .addBands([srtm, slope, aspect, entropia]) \
        .clip(moz)

    # Imputação de dados ausentes >5%
    mask_validos = imagem_final.mask().reduce(ee.Reducer.min())
    area_total = mask_validos.reduceRegion(**{
        'reducer': ee.Reducer.count(),
        'geometry': moz.geometry(),
        'scale': 30,
        'maxPixels': 1e13
    }).values().get(0)

    area_dados = imagem_final.select(0).reduceRegion(**{
        'reducer': ee.Reducer.count(),
        'geometry': moz.geometry(),
        'scale': 30,
        'maxPixels': 1e13
    }).values().get(0)

    proporcao = ee.Number(area_dados).divide(ee.Number(area_total))

    media_bandas = imagem_final.reduceRegion(**{
        'reducer': ee.Reducer.mean(),
        'geometry': moz.geometry(),
        'scale': 30,
        'maxPixels': 1e13
    })

    imagem_final = ee.Image(ee.Algorithms.If(
        proporcao.lt(0.95),
        imagem_final.unmask(ee.Image.constant(0).rename(imagem_final.bandNames()).addBands(media_bandas.toImage())),
        imagem_final
    ))

    # KMeans
    amostras = imagem_final.sample(**{
        'region': moz,
        'scale': 30,
        'numPixels': 5000,
        'seed': 42,
        'geometries': False
    })

    clusterizador = ee.Clusterer.wekaKMeans(6).train(**{
        'features': amostras,
        'inputProperties': imagem_final.bandNames()
    })

    resultado = imagem_final.cluster(clusterizador)

    # Exportação
    task = ee.batch.Export.image.toDrive(**{
        'image': resultado,
        'description': f'Cluster_{ano}_Mozambique',
        'folder': 'GEE',
        'fileNamePrefix': f'Cluster_{ano}',
        'region': moz.geometry(),
        'scale': 30,
        'maxPixels': 1e13
    })
    task.start()
    print(f'Tarefa de exportação enviada para {ano}.')

print("Processamento concluído.")


Processando 1991...
Tarefa de exportação enviada para 1991.
Processando 1996...
Tarefa de exportação enviada para 1996.
Processando 2001...
Tarefa de exportação enviada para 2001.
Processando 2006...
Tarefa de exportação enviada para 2006.
Processando 2011...
Tarefa de exportação enviada para 2011.
Processando 2016...
Tarefa de exportação enviada para 2016.
Processando 2021...
Tarefa de exportação enviada para 2021.
Processamento concluído.
